In [5]:
from helpers import Helpers
from entities import Element
from playwright.async_api import async_playwright
from langchain_core.tools import tool
import json
from dataclasses import dataclass
from typing import Any

from deepagents import create_deep_agent
from pywin.framework.toolmenu import tools
from sqlalchemy.sql.base import elements

from models import model, strong_model

SYSTEM_PROMPT = (
    "You are a web browser agent. Use `observe_page` to inspect the current page "
    "and available interactions. Given the user's task, suggest the"
    "next interaction steps. Base your suggestion only on the observed page state and "
    "provided context. Never assume an action has already occurred unless explicitly indicated."
)


@tool
async def observe_page() -> str:
    """ Observe the page interactions, it returns a string of the available interactions """
    with open("../js/scan-page.js", "r", encoding="utf-8") as f:
        scan_page_js = f.read()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto("https://backlogr.dev")
        result = await page.evaluate(scan_page_js)
        await browser.close()

        result_json = json.loads(result)
        output = Helpers.format_page_to_llm_output(result_json)
        return output


agent = create_deep_agent(model=strong_model, system_prompt=SYSTEM_PROMPT, tools=[observe_page])
result = await agent.ainvoke({"messages": [{"role": "user", "content": "Perform user login"}]})
print(result["messages"][-1].content)

# tool to observe

Enter your email and password in the respective fields, then select **Sign in**.
